# Homework  2 - Statistical Fairness Notions

**Due date: 3/3/2026**

In this homework, you will explore algorithmic fairness through recidivism prediction, examining how machine learning models can perpetuate or mitigate bias across demographic groups. We will be working again with the [COMPAS dataset](https://mlr3fairness.mlr-org.com/reference/compas.html) to understand disparities in prediction accuracy and implement key fairness metrics including demographic parity, equal opportunity, and equalized odds.

We will use FairLearn, an open-source, community-driven project to address fairness of AI systems. https://fairlearn.org/v0.13/quickstart.html



**Submission Requirements**<br>
Export your completed analysis as a PDF that clearly displays all code, outputs, and visualizations. Verify the PDF renders properly with readable font sizes and no truncated content. Include all discussion responses and ensure mathematical notation displays correctly. Submit the PDF and the Python Notebook (.ipynb) on Gradescope before the specified deadline.

# 1. Preparing the Data
We will just use a smaller subset of the data for this exercise. Please filter the data to keep the features and target specified below:

## Numeric Features:

- `age`

- `priors_count`

- `juv_fel_count`

- `juv_misd_count`

- `juv_other_count`

- `days_b_screening_arrest`

- `decile_score`


## Categorical Features:


#### Protected/Sensitive Feature

- `race`

- `sex`

#### Other

- `c_charge_degree`

- `score_text`


## Target:

- `two_year_recid`

For this honework, we will consider `race` as the protected attribute, so to simplify the analysis, please filter the data to just contain `Caucasian` and `African American` as the target subgroups for our fairness analysis.


You will also need to install the [Fairlearn](https://github.com/fairlearn/fairlearn?tab=readme-ov-file#getting-started-with-fairlearn) package for computing the fairness metrics. The cell below will download the package for you.

In [1]:
! pip install fairlearn

zsh:1: /Users/shantellluna/Downloads/Ethics-In-Artificial-Intelligence/.venv/bin/pip: bad interpreter: /Users/shantellluna/Downloads/Ethics_AI/.venv/bin/python3.13: no such file or directory


In [2]:
# Necessary imports
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import sklearn
import seaborn as sns

In [3]:
# Data source: ProPublica's COMPAS Analysis
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
data = pd.read_csv(url)

data = data.dropna(subset= 
        ['age',
        'priors_count',
        'juv_fel_count',
        'juv_misd_count',
        'juv_other_count',
        'days_b_screening_arrest',
        'decile_score'])

y = data['two_year_recid']
X = data.drop(columns=['two_year_recid'])

categorical_features = ['sex', 'race', 'age_cat', 'c_charge_degree']

numerical_features = [
    'age',
    'priors_count',
    'juv_fel_count',
    'juv_misd_count',
    'juv_other_count',
    'days_b_screening_arrest',
    'decile_score'
]

preprocessor = sklearn.compose.ColumnTransformer(
    transformers=[
        ('cat', sklearn.preprocessing.OneHotEncoder(drop="first"), categorical_features),
        ('num', sklearn.preprocessing.StandardScaler(), numerical_features)
    ]
)

# 2. Model Training

Before computing fairness metrics, you need to train a baseline classifier to generate predictions.

1. Train a logistic regression model using all the features. Make sure to encode your features as needed (numeric, categorical, ordinal), and don't forget to split the data into a train-test (70:30) before fitting the model.

2. Make predictions on the test set to generate `y_pred`. We will need this later for computing fairness metrics.

In [4]:
from sklearn.linear_model import LogisticRegression

# TODO: Train a logistic regression model
X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X_processed, y, test_size=0.3, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# TODO: Generate predictions on test set
y_pred = model.predict(X_test)

# 3. Fairness Metrics

Now that we have trained a model, let's check it against fairness metrics.

### 3.1 **Independence (Demographic Parity)**

A binary classifier satisfies independence/demographic parity if the following holds:

$$\Pr (\hat{Y} = 1 \mid A = a) = P (\hat{Y} = 1 \mid A = b) $$

for protected attributes $a, b \in A$. This implies that the probability of a positive prediction should be equal across the sensitive subgroups.

Use the `Fairlearn` package to compute the [demographic parity difference](https://fairlearn.org/v0.13/api_reference/generated/fairlearn.metrics.demographic_parity_difference.html) between the African American and Caucasian groups.

In [5]:
from fairlearn.metrics import demographic_parity_difference

# TODO: compute demographic parity by race
feature_names = preprocessor.get_feature_names_out()
race_index = list(feature_names).index('cat__race_Caucasian')

race_test = X_test[:, race_index]
dp_difference = demographic_parity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=race_test)

print("Demographic Parity Difference:", dp_difference)


Demographic Parity Difference: 0.15106539526195845


### 3.2 **Separation (Equal Opportunity)**

A binary classifier satisfies equal opportunity if the following holds:

$$ \Pr(\hat{Y} = 1 \mid Y = 1, A = a) = \Pr(\hat{Y} = 1 \mid Y = 1, A = b) $$

for protected attributes $a, b  \in A$. This is equivalent to saying that true positive rates (TPR) are equal across the sensitive subgroups. This also implies equal false negative rates (FNR) across groups, since FNR = 1 - TPR.

Use the `Fairlearn` package to compute the [equal opportunity difference](https://fairlearn.org/v0.13/api_reference/generated/fairlearn.metrics.equal_opportunity_difference.html) between the African American and Caucasian groups.

In [6]:
from fairlearn.metrics import equal_opportunity_difference

# TODO: compute equal opportunity by race
feature_names = preprocessor.get_feature_names_out()
race_index = list(feature_names).index('cat__race_Caucasian')

race_test = X_test[:, race_index]

eop_difference = equal_opportunity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=race_test)

print("Equal Opportunity Difference:", eop_difference)

Equal Opportunity Difference: 0.15629239614962198


### 3.3 **Separation (Equalized Odds)**

A binary classifier satisfies equalized odds if the true positive rate (TPR) and false positive rate (FPR) is equal across protected attribute groups $a, b \in A$:

$$ \Pr(\hat{Y} = 1 \mid Y = 1, A = a) = \Pr(\hat{Y} = 1 \mid Y = 1, A = b) \qquad \text{(Equal TPR) }$$
$$ \Pr(\hat{Y} = 1 \mid Y = 0, A = a) = \Pr(\hat{Y} = 1 \mid Y = 0, A = b) \qquad \text{  (Equal FPR)} $$

This criterion combines equal opportunity, which requires only equal TPRs across subgroups, with an additional requirement of equal FPRs, ensuring that both benefits (true positives) and harms (false positives) of prediction are distributed fairly across groups.

Use the `Fairlearn` package to compute the [equalized odds difference](https://fairlearn.org/v0.13/api_reference/index.html) between the African American and Caucasian groups.

In [7]:
from fairlearn.metrics import equalized_odds_difference

# TODO: compute equalized odds by race
feature_names = preprocessor.get_feature_names_out()
race_index = list(feature_names).index('cat__race_Caucasian')

race_test = X_test[:, race_index]

eod_difference = equalized_odds_difference(y_true=y_test, y_pred=y_pred, sensitive_features=race_test)

print("Equalized Odds Difference:", eod_difference)

Equalized Odds Difference: 0.15629239614962198


### 3.4 **Sufficiency (Predictive Parity)**

A binary classifier satisfies the sufficiency condition (predictive parity) if the following holds:

$$ \Pr(Y = 1 \mid \hat{Y} = \hat{y}, A = a) =  \Pr(Y = 1 \mid \hat{Y} = \hat{y}, A = b) $$

for protected attributes $a, b \in A$ and for $\hat{y} \in \{0,1\}$. This criterion ensures equal precision across sensitive subgroups.

Use the `Fairlearn` package to compute the predictive parity difference between the African American and Caucasian groups. You will need to use the [`MetricsFrame`](https://fairlearn.org/v0.13/api_reference/generated/fairlearn.metrics.MetricFrame.html) object and specify the [`precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html) method as the metric to achieve this.

In [8]:
from fairlearn.metrics import MetricFrame
from sklearn.metrics import precision_score

# TODO: compute predictive parity by race
feature_names = preprocessor.get_feature_names_out()
race_index = list(feature_names).index('cat__race_Caucasian')

race_test = X_test[:, race_index]

metric_frame = MetricFrame(metrics=precision_score, y_true=y_test, y_pred=y_pred, sensitive_features=race_test)

predictive_parity = metric_frame.difference()

print("Predictive Parity Difference:", predictive_parity)

Predictive Parity Difference: 0.04369913257656888


# 4. Fairness through Unawareness

In this question we will explore why simply removing a sensitive attribute may not guarantee the impartiality of the resulting classifier. You will investigate the phenomenon of proxy encodings — cases where non-sensitive features are highly correlated with the protected attribute(s), potentially leading to discriminatory outcomes even when the protected attribute is not directly used.


1. Compute the correlation between each feature and the protected attribute `race`. Which feature(s) do you see are most associated with race?

   **Hint:** You might want to standardize numeric features before computing the correlations.

   **Hint:** Use `.corr()` on the processed features to easily get the correlation dataframe, assuming you keep your features as a Pandas dataframe after processing.

2. Can you predict race from the remaining features (excluding the target)?

   - Train a logistic regression model to predict race using all other features
   - Report the accuracy and identify the top 3 most predictive features

3. Train two logistic regression models to predict recidivism:
    - **Model A:** Include all features **including** `race`
    - **Model B:** Include all features **except** `race`

4. Report the accuracy and the four fairness metrics from Part 3 for each model with respect to `race`.


In [9]:
# TODO: compute correlation between race and other features
feature_names = preprocessor.get_feature_names_out()
X_df = pd.DataFrame(X_processed, columns=feature_names)

race_corr = race_correlations = X_df.corr()["cat__race_Caucasian"].sort_values(ascending=False)

print(race_corr)

cat__race_Caucasian             1.000000
num__age                        0.181719
cat__age_cat_Greater than 45    0.157172
cat__c_charge_degree_M          0.069338
num__juv_other_count           -0.023165
num__days_b_screening_arrest   -0.032744
cat__race_Native American      -0.034916
cat__race_Asian                -0.049436
num__juv_fel_count             -0.061968
cat__sex_Male                  -0.066331
num__juv_misd_count            -0.071994
cat__age_cat_Less than 25      -0.098983
num__priors_count              -0.131783
cat__race_Other                -0.169916
num__decile_score              -0.192826
cat__race_Hispanic             -0.220216
Name: cat__race_Caucasian, dtype: float64


In [10]:
from sklearn.metrics import accuracy_score

# TODO: predict race from remaining features (excluding target)
y_race = data["race"]

features = [
    "age",
    "priors_count",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "days_b_screening_arrest",
    "decile_score",
    "sex",
    "c_charge_degree",
    "score_text"
]

X_race = data[features]

X_race = pd.get_dummies(X_race, drop_first=True)

X_train_r, X_test_r, y_train_r, y_test_r = sklearn.model_selection.train_test_split(X_race, y_race, test_size=0.3, random_state=42)

race_model = LogisticRegression(max_iter=1000)
race_model.fit(X_train_r, y_train_r)

race_pred = race_model.predict(X_test_r)

race_accuracy = accuracy_score(y_test_r, race_pred)

print("Race prediction accuracy:", race_accuracy)

Race prediction accuracy: 0.5634346357935359


/Users/shantellluna/Downloads/Ethics-In-Artificial-Intelligence/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
# top 3 features
coefficients = pd.Series(race_model.coef_[0], index=X_race.columns)
top_features = coefficients.abs().sort_values(ascending=False).head(3)

print("Top 3 most predictive features:")
print(top_features)

Top 3 most predictive features:
score_text_Low       0.505944
decile_score         0.247465
score_text_Medium    0.111721
dtype: float64


In [12]:
# TODO: Build Model A and Model B
# Model A
X_A = data.drop(columns=["two_year_recid"])
y_A = data["two_year_recid"]

X_A_processed = preprocessor.fit_transform(X_A)

X_train_A, X_test_A, y_train_A, y_test_A = sklearn.model_selection.train_test_split(X_A_processed, y_A, test_size=0.3, random_state=42)

model_A = LogisticRegression(max_iter=1000)
model_A.fit(X_train_A, y_train_A)

y_pred_A = model_A.predict(X_test_A)

accuracy_A = accuracy_score(y_test_A, y_pred_A)



race_test_A = X_test_A[:, race_index]

dp_A = demographic_parity_difference(y_test_A, y_pred_A, sensitive_features=race_test_A)
eo_A = equal_opportunity_difference(y_test_A, y_pred_A, sensitive_features=race_test_A)
eod_A = equalized_odds_difference(y_test_A, y_pred_A, sensitive_features=race_test_A)

mf_A = MetricFrame(metrics=precision_score, y_true=y_test_A, y_pred=y_pred_A, sensitive_features=race_test_A)

pp_A = mf_A.difference()

In [13]:
print("Model A Accuracy:", accuracy_A)
print("Model A Fairness:")
print("DP:", dp_A)
print("EO:", eo_A)
print("EOD:", eod_A)
print("PP:", pp_A)

Model A Accuracy: 0.6816208393632417
Model A Fairness:
DP: 0.15106539526195845
EO: 0.15629239614962198
EOD: 0.15629239614962198
PP: 0.04369913257656888


In [14]:
# Model B
X_B = data.drop(columns=["two_year_recid", "race"])
y_B = data["two_year_recid"]

categorical_features_B = ["sex", "age_cat", "c_charge_degree"]

preprocessor_B = sklearn.compose.ColumnTransformer(
    transformers=[
        ("cat", sklearn.preprocessing.OneHotEncoder(drop="first"), categorical_features_B),
        ("num", sklearn.preprocessing.StandardScaler(), numerical_features),
    ]
)

X_B_processed = preprocessor_B.fit_transform(X_B)

X_train_B, X_test_B, y_train_B, y_test_B = sklearn.model_selection.train_test_split(X_B_processed, y_B, test_size=0.3, random_state=42)

model_B = LogisticRegression(max_iter=1000)
model_B.fit(X_train_B, y_train_B)

y_pred_B = model_B.predict(X_test_B)

accuracy_B = accuracy_score(y_test_B, y_pred_B)



race_test_B = data.loc[y_test_B.index, "race"]

dp_B = demographic_parity_difference(y_test_B, y_pred_B, sensitive_features=race_test_B)

eo_B = equal_opportunity_difference(y_test_B, y_pred_B, sensitive_features=race_test_B)

eod_B = equalized_odds_difference(y_test_B, y_pred_B, sensitive_features=race_test_B)

mf_B = MetricFrame(metrics=precision_score, y_true=y_test_B, y_pred=y_pred_B, sensitive_features=race_test_B)

pp_B = mf_B.difference()

In [15]:
print("Model B Accuracy:", accuracy_B)
print("Model B Fairness:")
print("DP:", dp_B)
print("EO:", eo_B)
print("EOD:", eod_B)
print("PP:", pp_B)

Model B Accuracy: 0.6811384466956102
Model B Fairness:
DP: 0.5138888888888888
EO: 0.6842105263157895
EOD: 0.6842105263157895
PP: 0.42000000000000004


In [16]:
!jupyter nbconvert hw_2.ipynb --to html --embed-images

zsh:1: /Users/shantellluna/Downloads/Ethics-In-Artificial-Intelligence/.venv/bin/jupyter: bad interpreter: /Users/shantellluna/Downloads/Ethics_AI/.venv/bin/python3.13: no such file or directory
